## 1. Setup

In [3]:
from pathlib import Path

In [4]:
base_url = "https://opendata.schleswig-holstein.de/dataset/fahrplandaten"
# "https://opendata.schleswig-holstein.de/dataset/5e0652d8-7f59-42fc-92ab-a6358f800e1d/resource/a2d65393-bb77-4e4c-abe3-9fe69c58facc/download/fahrplandaten"
out_dir = Path("../data/fahrplandaten/")

# 2. Download Data

In [5]:
import requests
from io import BytesIO
from zipfile import ZipFile
import os
from bs4 import BeautifulSoup

In [6]:
# Create out directory
os.makedirs(out_dir, exist_ok=True)

# Download and unpack
response = requests.get(base_url)
response.raise_for_status()

# Get link from webpage
bs = BeautifulSoup(response.content, "html.parser")
data_endpoint = bs.select_one("div.resource-title a")["href"]
data_url = "/".join(base_url.split("/")[:-2]) + data_endpoint

# Download data at link
response = requests.get(data_url)
response.raise_for_status()

# Unpack response
with ZipFile(BytesIO(response.content)) as zip_ref:
    zip_ref.extractall(out_dir)

print(f"Extracted to {out_dir}.")

del bs, data_endpoint, data_url
del response
del zip_ref

Extracted to ../data/fahrplandaten.


# 3. Query Data

In [7]:
import pandas as pd

In [8]:
# Collect all dataframes
def load_df(name: str, verbose=True) -> pd.DataFrame:
    df = pd.read_csv(out_dir / f"{name}.txt")
    if verbose:
        print(f"df_{name}:\t{df.columns.to_list()}")
    return df

df_agency = load_df("agency")
df_calendar = load_df("calendar")
df_calendar_dates = load_df("calendar_dates")
# df_frequencies = load_df("frequencies")
df_routes = load_df("routes")
# df_service_alerts = load_df("service_alerts")
df_shapes = load_df("shapes")
df_stop_times = load_df("stop_times")
df_stops = load_df("stops")
df_transfers = load_df("transfers")
df_trips = load_df("trips")

df_agency:	['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']
df_calendar:	['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
df_calendar_dates:	['service_id', 'date', 'exception_type']
df_routes:	['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'route_desc']
df_shapes:	['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']
df_stop_times:	['trip_id', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'stop_headsign', 'arrival_time', 'departure_time']
df_stops:	['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code', 'zone_id']
df_transfers:	['from_stop_id', 'to_stop_id', 'transfer_type', 'min_transfer_time', 'from_route_id', 'to_route_id', 'from_trip_id', 'to_trip_id']
df_trips:	['route_id', 'service_id

/tmp/ipykernel_389627/1544208904.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out_dir / f"{name}.txt")


In [9]:
""" Filter dataframes for KVG data """

# Get KVG agency id
kvg_id = df_agency[df_agency.agency_name == "Kieler Verkehrsgesellschaft mbH"].agency_id.iloc[0]
# print(kvg_id)

# Filter routes to kvg-only
df_kvg_routes = df_routes[df_routes.agency_id == kvg_id].loc[:, ["route_id", "route_short_name"]]
# print(df_kvg_routes)

# Filter trips to kvg-only
df_kvg_trips = df_trips[df_trips.route_id.isin(df_kvg_routes.route_id)].loc[:, ["route_id", "trip_id", "trip_headsign", "service_id"]]

# Filter calendar dates
# df_kvg_calendar = df_calendar[df_calendar.service_id.isin(df_kvg_trips.service_id)]  # every day entry is 0
df_kvg_calendar_dates = df_calendar_dates[df_calendar_dates.service_id.isin(df_kvg_trips.service_id)].loc[:, ["service_id", "date"]]
df_kvg_calendar_dates.date = pd.to_datetime(df_kvg_calendar_dates.loc[:, "date"], format="%Y%m%d")

# Filter stop times to kvg-only
df_kvg_stop_times = df_stop_times[df_stop_times.trip_id.isin(df_kvg_trips.trip_id)].loc[:, ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"]]

# Filter stops by kvg-only
df_kvg_stops = df_stops[df_stops.stop_id.isin(pd.unique(df_kvg_stop_times.stop_id))].loc[:, ["stop_id", "stop_name", "stop_lat", "stop_lon"]]


In [10]:
""" Enhance data """

# Arrival and departure times to timedeltas
df_kvg_stop_times.arrival_time = pd.to_timedelta(df_kvg_stop_times.arrival_time)
df_kvg_stop_times.departure_time = pd.to_timedelta(df_kvg_stop_times.departure_time)

# Add dates to stop times
df_kvg_dates = df_kvg_trips.merge(df_calendar_dates, on="service_id").loc[:, ["trip_id", "date"]]
df_kvg_dates.date = pd.to_datetime(df_kvg_dates.date, format="%Y%m%d")
df_kvg_stop_times = df_kvg_stop_times.merge(df_kvg_dates, on="trip_id")

# Combine date and time
df_kvg_stop_times.arrival_time = df_kvg_stop_times.date + df_kvg_stop_times.arrival_time
df_kvg_stop_times.departure_time = df_kvg_stop_times.date + df_kvg_stop_times.departure_time

In [11]:
""" Combine dataframes """

df_kvg = df_kvg_stop_times \
    .merge(df_kvg_stops, on="stop_id") \
    .merge(df_kvg_trips, on="trip_id") \
    .merge(df_kvg_routes, on="route_id") \
    .loc[:, ["route_short_name", "stop_sequence", "arrival_time", "departure_time", "stop_name", "stop_lon", "stop_lat"]]

"""
# routes.route_short_name   stop_times.stop_sequence    stop_times.arrival_time stop_times.departure_time   stops.stop_name stops.stop_lon  stops.stop_lat  
# 91                        0..n                        2026-01-01 12:00:00     2026-01-01 12:05_00         Hauptbahnhof    52.96           11.08           
"""
df_kvg

,route_short_name,stop_sequence,arrival_time,departure_time,stop_name,stop_lon,stop_lat
0,X91,0,2026-09-19 10:52:00,2026-09-19 10:52:00,Kiel Hauptbahnhof,10.130686,54.315500
1,X91,0,2026-11-07 10:52:00,2026-11-07 10:52:00,Kiel Hauptbahnhof,10.130686,54.315500
2,X91,1,2026-09-19 10:57:00,2026-09-19 10:57:00,Kiel Wilhelmplatz,10.120359,54.322625
3,X91,1,2026-11-07 10:57:00,2026-11-07 10:57:00,Kiel Wilhelmplatz,10.120359,54.322625
4,X91,2,2026-09-19 10:59:00,2026-09-19 10:59:00,Kiel Gravelottestraße,10.113324,54.325003
...,...,...,...,...,...,...,...
7232783,11,32,2026-10-12 00:09:00,2026-10-12 00:09:00,Kiel Wik Kanal,10.131386,54.365402
7232784,11,32,2026-10-19 00:09:00,2026-10-19 00:09:00,Kiel Wik Kanal,10.131386,54.365402
7232785,11,32,2026-10-26 00:09:00,2026-10-26 00:09:00,Kiel Wik Kanal,10.131386,54.365402
7232786,11,32,2026-11-02 00:09:00,2026-11-02 00:09:00,Kiel Wik Kanal,10.131386,54.365402


In [12]:
""" Filter data """
start_date = "2026-11-05"
end_date = "2026-11-06"
df_kvg = df_kvg[(df_kvg.departure_time > start_date) & (df_kvg.arrival_time < end_date)]
df_kvg.sort_values(by="arrival_time")

,route_short_name,stop_sequence,arrival_time,departure_time,stop_name,stop_lon,stop_lat
5231,N62,20,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
240527,N14,20,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
202423,N22,18,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
263835,N14,35,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
292871,N12,35,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
...,...,...,...,...,...,...,...
5974490,14,52,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Stavangerstraße,10.050259,54.319502
4630418,31,33,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Isarweg,10.193354,54.289073
2948398,45,6,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel TÜV,10.151193,54.297244
7232170,11,22,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Schauenburgerstraße,10.132618,54.332798


# 4. Save data

In [13]:
import os

output_file = "../data/kvg_data.csv"


os.makedirs(os.path.dirname(output_file), exist_ok=True)
df_kvg.to_csv(output_file, index=False)